# vLLM for a Multi-Agent RAG System: Production Inference Guide

One notebook that walks through the techniques vLLM uses to keep a GPU busy, and how to tune them for a **multi-agent RAG workload on a local server**.

| # | Topic | What you learn |
|---|-------|----------------|
| 1 | KV-cache math | Why memory (not compute) limits concurrency |
| 2 | PagedAttention | Block tables, no fragmentation, prefix sharing (toy implementation) |
| 3 | Continuous batching | Static vs. continuous scheduling (toy simulation) |
| 4 | Real vLLM engine | The knobs that control utilization, throughput, latency |
| 5 | Prefix caching | Huge win for RAG: shared system prompts / agent prompts / repeated docs |
| 6 | Chunked prefill | Keeps decode latency stable while long RAG prompts arrive |
| 7 | Production server | `vllm serve`, async clients, streaming, TTFT |
| 8 | Multi-agent patterns | Priorities, structured outputs, LoRA per agent, concurrency control |
| 9 | Speculative decoding and quantization | Lower latency / more memory headroom |
| 10 | Observability | Prometheus metrics to know when to tune |
| 11 | Benchmarking and tuning checklist | Measure, change one knob, re-measure |

> **Note:** vLLM evolves quickly. Flag names and metric names shift between versions. Run `vllm serve --help` and check the docs for your installed version. Sections 1-3 are pure Python and run anywhere; sections 4+ need an NVIDIA GPU.

In [ ]:
# Setup (run once). Pin versions in production.
# !pip install -U vllm openai aiohttp numpy pandas matplotlib

import subprocess, shutil
print(shutil.which("nvidia-smi") and subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
    capture_output=True, text=True).stdout)
try:
    import vllm; print("vllm", vllm.__version__)
except ImportError:
    print("vllm not installed; sections 1-3 still work")

## 1. KV-cache math: why memory limits concurrency

During generation every token keeps its **Key** and **Value** vectors for every layer so attention doesn't recompute them. That is the KV cache.

```
bytes per token = 2 (K and V) x num_layers x num_kv_heads x head_dim x bytes_per_element
```

GPU memory splits roughly into: **model weights + activations/workspace + KV cache**. vLLM gives whatever is left (within `gpu_memory_utilization`) to the KV cache. More KV cache means more concurrent sequences means higher throughput.

In [ ]:
def kv_bytes_per_token(num_layers, num_kv_heads, head_dim, dtype_bytes=2):
    return 2 * num_layers * num_kv_heads * head_dim * dtype_bytes

def capacity_report(name, num_layers, num_kv_heads, head_dim, gpu_gb, weights_gb,
                    gpu_util=0.90, overhead_gb=2.0, dtype_bytes=2, avg_ctx=4096):
    per_tok = kv_bytes_per_token(num_layers, num_kv_heads, head_dim, dtype_bytes)
    kv_gb = gpu_gb * gpu_util - weights_gb - overhead_gb
    total_tokens = int(kv_gb * 1024**3 / per_tok)
    print(f"{name}")
    print(f"  KV per token       : {per_tok/1024:.1f} KB")
    print(f"  KV cache budget    : {kv_gb:.1f} GB  -> {total_tokens:,} cacheable tokens")
    print(f"  Concurrent seqs @ {avg_ctx} ctx: ~{total_tokens // avg_ctx}")
    return total_tokens

# Example: Qwen2.5-7B-Instruct (28 layers, 4 KV heads via GQA, head_dim 128), 24 GB GPU, fp16 weights ~15 GB
capacity_report("Qwen2.5-7B fp16 KV, 24GB GPU", 28, 4, 128, gpu_gb=24, weights_gb=15.2)
# FP8 KV cache halves per-token cost -> ~2x tokens
capacity_report("Qwen2.5-7B fp8 KV,  24GB GPU", 28, 4, 128, gpu_gb=24, weights_gb=15.2, dtype_bytes=1)

## 2. PagedAttention: the idea

Naive servers reserve one contiguous KV buffer per request sized for `max_len`. That causes **internal fragmentation** (reserved but unused) and **external fragmentation** (holes between buffers). Typically 60-80% of KV memory is wasted.

PagedAttention borrows OS virtual memory ideas:

- KV cache is split into fixed-size **blocks** (default 16 tokens in many versions; `--block-size`).
- Each sequence has a **block table** (logical block to physical block).
- Blocks are allocated on demand as tokens are generated, so waste is at most one partially-filled block per sequence.
- Blocks can be **shared** between sequences (same prefix, parallel sampling, beam search) with reference counting and copy-on-write.

Below is a toy block manager to build intuition. It is not vLLM's code, just the concept.

In [ ]:
import math
from collections import defaultdict

class ToyBlockManager:
    def __init__(self, num_blocks, block_size=16):
        self.block_size = block_size
        self.free = list(range(num_blocks))
        self.ref = defaultdict(int)          # physical block -> refcount
        self.tables = {}                      # seq_id -> [physical blocks]
        self.lens = {}                        # seq_id -> tokens stored

    def _alloc(self):
        if not self.free:
            raise MemoryError("Out of KV blocks -> scheduler must preempt/queue")
        b = self.free.pop(); self.ref[b] = 1; return b

    def add_sequence(self, seq_id, prompt_len):
        n = math.ceil(prompt_len / self.block_size)
        self.tables[seq_id] = [self._alloc() for _ in range(n)]
        self.lens[seq_id] = prompt_len

    def append_token(self, seq_id):
        if self.lens[seq_id] % self.block_size == 0:      # last block full -> new block
            self.tables[seq_id].append(self._alloc())
        else:                                              # writing into last block
            last = self.tables[seq_id][-1]
            if self.ref[last] > 1:                         # shared -> copy-on-write
                self.ref[last] -= 1
                self.tables[seq_id][-1] = self._alloc()
        self.lens[seq_id] += 1

    def fork(self, parent, child):
        """Share all blocks (e.g., same RAG prefix or n>1 sampling)."""
        self.tables[child] = list(self.tables[parent])
        self.lens[child] = self.lens[parent]
        for b in self.tables[child]: self.ref[b] += 1

    def free_sequence(self, seq_id):
        for b in self.tables.pop(seq_id):
            self.ref[b] -= 1
            if self.ref[b] == 0: self.free.append(b)
        self.lens.pop(seq_id)

    def stats(self):
        used = len({b for t in self.tables.values() for b in t})
        return {"used_blocks": used, "free_blocks": len(self.free)}

bm = ToyBlockManager(num_blocks=64, block_size=16)
bm.add_sequence("rag_prefix", prompt_len=100)       # shared system prompt + retrieved docs
print("after prefix        :", bm.stats(), bm.tables["rag_prefix"])
bm.fork("rag_prefix", "agentA"); bm.fork("rag_prefix", "agentB")
print("after 2 forks       :", bm.stats(), "(no new memory!)")
for _ in range(30): bm.append_token("agentA")
for _ in range(5):  bm.append_token("agentB")
print("after divergence    :", bm.stats())
print("agentA table", bm.tables["agentA"])
print("agentB table", bm.tables["agentB"])

# Fragmentation comparison: contiguous max_len reservation vs paged
max_len, actual = 4096, [300, 700, 1200, 250, 900]
naive_tokens = max_len * len(actual)
paged_tokens = sum(math.ceil(a/16)*16 for a in actual)
print(f"\nContiguous reserve: {naive_tokens} tokens | Paged: {paged_tokens} tokens | "
      f"saved {100*(1-paged_tokens/naive_tokens):.0f}%")

## 3. Continuous (iteration-level) batching

**Static batching:** group N requests, run until *all* finish. Short requests sit idle waiting for the longest, and new requests wait for the whole batch.

**Continuous batching:** the scheduler makes a decision at **every decode step**. Finished sequences leave immediately and waiting requests join immediately, so GPU slots stay full.

In vLLM the scheduler also: admits requests while KV blocks are available, **preempts** (recompute or swap) when KV cache is exhausted, and mixes prefill and decode work in the same step (see chunked prefill, section 6).

Simulation: each step = one forward pass producing one token per running sequence.

In [ ]:
import random, numpy as np
import matplotlib.pyplot as plt

random.seed(0)
N, SLOTS = 200, 16
out_lens = [int(random.lognormvariate(4.5, 0.9)) + 5 for _ in range(N)]   # heavy-tailed output lengths

def static_batching(lens, slots):
    steps, useful = 0, 0
    for i in range(0, len(lens), slots):
        batch = lens[i:i+slots]
        steps += max(batch)                    # everyone waits for the longest
        useful += sum(batch)
    return steps, useful / (steps * slots)

def continuous_batching(lens, slots):
    queue, running, steps, useful = list(lens), [], 0, 0
    while queue or running:
        while queue and len(running) < slots:  # refill at EVERY step
            running.append(queue.pop(0))
        steps += 1; useful += len(running)
        running = [r-1 for r in running if r-1 > 0]
    return steps, useful / (steps * slots)

s_steps, s_util = static_batching(out_lens, SLOTS)
c_steps, c_util = continuous_batching(out_lens, SLOTS)
print(f"static     : {s_steps:5d} steps, slot utilization {s_util:.0%}")
print(f"continuous : {c_steps:5d} steps, slot utilization {c_util:.0%}")
print(f"throughput gain: {s_steps/c_steps:.2f}x")

plt.bar(["static", "continuous"], [s_util*100, c_util*100]); plt.ylabel("GPU slot utilization (%)")
plt.title("Static vs continuous batching (toy)"); plt.show()

## 4. The real vLLM engine and the knobs that matter

| Parameter | What it controls | Tuning guidance |
|-----------|------------------|-----------------|
| `gpu_memory_utilization` | Fraction of GPU memory vLLM may use (weights + KV) | 0.85-0.92. Higher = more KV cache but OOM risk if other processes share the GPU |
| `max_model_len` | Max prompt+output per request | Set to what you really need. Smaller = less reserved, easier to fit many seqs |
| `max_num_seqs` | Max concurrent sequences in a step | Raise for throughput (128-512) if KV cache allows; lower for latency |
| `max_num_batched_tokens` | Token budget per scheduler step | Higher = better throughput, higher inter-token latency. 2048-16384 typical |
| `enable_prefix_caching` | Reuse KV of identical prompt prefixes | Big win for RAG/agents with shared prefixes |
| `enable_chunked_prefill` | Split long prefills into chunks mixed with decodes | Smooths latency; default on in recent V1 engine |
| `kv_cache_dtype` | `auto`, `fp8` | `fp8` ~2x KV capacity, small quality cost |
| `quantization` | AWQ / GPTQ / FP8 etc. | Frees memory for KV cache; FP8 is great on Hopper/Ada |
| `tensor_parallel_size` | Shard model across GPUs | Use when model doesn't fit one GPU; adds comms overhead |
| `swap_space` | CPU RAM (GB) for swapped KV (older engines) | Mostly relevant to older versions |

First, a basic offline run with `LLM` (this is the engine under the hood of the server too).

In [ ]:
from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen2.5-7B-Instruct"   # swap for your local path, e.g. "/models/qwen2.5-7b-instruct"

llm = LLM(
    model=MODEL,
    dtype="auto",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    max_num_seqs=128,
    max_num_batched_tokens=8192,
    enable_prefix_caching=True,
)

params = SamplingParams(temperature=0.2, top_p=0.9, max_tokens=128)
prompts = [f"Summarize policy #{i} in one sentence." for i in range(8)]
outs = llm.generate(prompts, params)       # vLLM batches these continuously for you
for o in outs[:2]:
    print(o.prompt, "->", o.outputs[0].text.strip()[:100])

### Measuring throughput properly
Always measure **tokens/sec**, **TTFT** (time to first token) and **inter-token latency**, not just wall time. Offline, the easiest signal is total generated tokens divided by wall time.

In [ ]:
import time

def offline_throughput(llm, n_prompts=256, max_tokens=128):
    prompts = [f"Write a short paragraph about topic number {i}." for i in range(n_prompts)]
    sp = SamplingParams(temperature=0.7, max_tokens=max_tokens, ignore_eos=True)
    t0 = time.perf_counter()
    outs = llm.generate(prompts, sp, use_tqdm=False)
    dt = time.perf_counter() - t0
    gen = sum(len(o.outputs[0].token_ids) for o in outs)
    print(f"{n_prompts} reqs | {gen} gen tokens | {dt:.1f}s | {gen/dt:,.0f} tok/s")

offline_throughput(llm)

## 5. Automatic prefix caching (huge for RAG and multi-agent)

vLLM hashes each KV block by its token content plus the prefix before it. If a new request begins with tokens already cached, those blocks are **reused instead of recomputed**, so prefill for that part is nearly free.

In a multi-agent RAG system, shared prefixes are everywhere:

- Each agent's long **system prompt** and tool schema
- Few-shot examples
- The same **retrieved document context** reused across follow-up questions
- Conversation history (each turn extends the previous prompt)

**Prompt design rule:** put stable content **first**, variable content **last**.

```
[system prompt][tool schema][few-shot]  <- cached across all requests
[retrieved docs]                         <- cached if same docs repeat
[user question]                          <- always new
```

Anything dynamic early in the prompt (timestamps, request IDs, user name) breaks the cache for everything after it.

Engine settings can't be changed in a live process reliably, so we benchmark in a **subprocess** per configuration.

In [ ]:
%%writefile /tmp/bench_prefix.py
import sys, json, time
from vllm import LLM, SamplingParams

cfg = json.loads(sys.argv[1])
llm = LLM(model=cfg["model"], gpu_memory_utilization=0.85, max_model_len=8192,
          enable_prefix_caching=cfg["prefix_caching"],
          enable_chunked_prefill=cfg.get("chunked_prefill", True),
          max_num_batched_tokens=cfg.get("max_batched", 8192))

shared_context = ("POLICY HANDBOOK EXCERPT. " + "Employees must follow rule section X. " * 400)  # ~3k tokens
questions = [f"Question {i}: what does section {i} say?" for i in range(64)]
prompts = [f"{shared_context}\n\n{q}" for q in questions]
sp = SamplingParams(temperature=0, max_tokens=32)

llm.generate(prompts[:2], sp, use_tqdm=False)          # warm-up
t0 = time.perf_counter()
llm.generate(prompts, sp, use_tqdm=False)
dt = time.perf_counter() - t0
print(json.dumps({"prefix_caching": cfg["prefix_caching"], "seconds": round(dt, 2)}))

In [ ]:
import json, subprocess, sys

def run(cfg):
    r = subprocess.run([sys.executable, "/tmp/bench_prefix.py", json.dumps(cfg)],
                       capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("{")]
    return json.loads(line[-1]) if line else r.stderr[-1500:]

# Free the GPU held by the notebook's engine before spawning subprocesses
import gc, torch
try:
    del llm
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

print(run({"model": MODEL, "prefix_caching": False}))
print(run({"model": MODEL, "prefix_caching": True}))

## 6. Chunked prefill

A 6k-token RAG prompt needs a big prefill. Without chunking, that single prefill hogs a whole step, so **every running decode stalls** (latency spike for other agents).

**Chunked prefill** splits the prompt into chunks of at most `max_num_batched_tokens` and mixes them with decode tokens in the same step. Decodes get scheduled first, then leftover token budget is filled with prefill chunks.

Tuning trade-off:

| `max_num_batched_tokens` | Effect |
|---|---|
| Small (512-2048) | Better inter-token latency (ITL), slower TTFT for long prompts |
| Large (8192-16384) | Better throughput and TTFT, more ITL jitter |

For interactive agents, start around 2048-4096. For batch or offline pipelines, go higher.

Compare settings with the same subprocess harness, reusing `bench_prefix.py` with a different `max_batched`:

In [ ]:
for mb in (1024, 4096, 16384):
    out = run({"model": MODEL, "prefix_caching": False, "chunked_prefill": True, "max_batched": mb})
    print("max_num_batched_tokens =", mb, "->", out)

## 7. Production serving: `vllm serve` (OpenAI-compatible)

For a multi-agent system you want **one long-lived server** that all agents call over HTTP, rather than each agent loading its own model. vLLM's continuous batching then merges requests from all agents into the same GPU steps automatically. That is where the utilization win comes from.

A solid starting command (single GPU, 7-8B model):

```bash
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --host 0.0.0.0 --port 8000 \
  --served-model-name rag-llm \
  --gpu-memory-utilization 0.90 \
  --max-model-len 8192 \
  --max-num-seqs 128 \
  --max-num-batched-tokens 4096 \
  --enable-prefix-caching \
  --enable-chunked-prefill \
  --api-key $VLLM_API_KEY
```

Multi-GPU: add `--tensor-parallel-size 2` (same node, NVLink preferred). To scale throughput for a model that already fits one GPU, run **one replica per GPU** behind a load balancer (nginx/HAProxy/LiteLLM) instead of tensor parallelism. Replicas scale nearly linearly; TP does not.

Launch from the notebook (or use systemd/Docker in real production):

In [ ]:
import subprocess, time, requests, os

SERVER_CMD = [
    "vllm", "serve", MODEL,
    "--host", "0.0.0.0", "--port", "8000",
    "--served-model-name", "rag-llm",
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "8192",
    "--max-num-seqs", "128",
    "--max-num-batched-tokens", "4096",
    "--enable-prefix-caching",
    "--scheduling-policy", "priority",     # used in section 8
]
server = subprocess.Popen(SERVER_CMD, stdout=open("/tmp/vllm_server.log", "w"), stderr=subprocess.STDOUT)

for _ in range(180):                       # wait until healthy
    try:
        if requests.get("http://localhost:8000/health", timeout=1).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/tmp/vllm_server.log").read()[-2000:])

### Docker and systemd (what you'd really deploy)

```bash
docker run -d --name vllm --gpus all --ipc=host -p 8000:8000 \
  -v /models:/models \
  vllm/vllm-openai:latest \
  --model /models/qwen2.5-7b-instruct --served-model-name rag-llm \
  --gpu-memory-utilization 0.90 --max-model-len 8192 \
  --enable-prefix-caching --max-num-seqs 128
```

Add `--restart unless-stopped`, a health check on `/health`, and pin the image tag instead of `latest`.

### Async client with streaming: measure TTFT and inter-token latency

Use `AsyncOpenAI` and `asyncio` so many agent calls are in flight at once. Sending requests one at a time wastes the GPU, because continuous batching only helps if requests overlap.

In [ ]:
import asyncio, time, statistics
from openai import AsyncOpenAI

client = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key=os.environ.get("VLLM_API_KEY", "EMPTY"))

async def one_request(prompt, max_tokens=128, priority=None):
    t0 = time.perf_counter(); first = None; n = 0
    extra = {"priority": priority} if priority is not None else None
    stream = await client.chat.completions.create(
        model="rag-llm", messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens, temperature=0.2, stream=True, extra_body=extra)
    async for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if first is None: first = time.perf_counter()
            n += 1
    end = time.perf_counter()
    return {"ttft": first - t0, "total": end - t0, "tokens": n,
            "itl": (end - first) / max(n - 1, 1)}

async def load_test(concurrency, n_requests=64, **kw):
    sem = asyncio.Semaphore(concurrency)
    async def guarded(i):
        async with sem:
            return await one_request(f"Explain topic {i} for a new employee.", **kw)
    t0 = time.perf_counter()
    res = await asyncio.gather(*[guarded(i) for i in range(n_requests)])
    wall = time.perf_counter() - t0
    toks = sum(r["tokens"] for r in res)
    p = lambda k, q: statistics.quantiles([r[k] for r in res], n=100)[q-1]
    print(f"conc={concurrency:3d} | {toks/wall:7.0f} tok/s | TTFT p50 {p('ttft',50)*1000:5.0f}ms "
          f"p95 {p('ttft',95)*1000:5.0f}ms | ITL p50 {p('itl',50)*1000:5.1f}ms")

# In a notebook, top-level await works
for c in (1, 8, 32, 64):
    await load_test(c)

**How to read this:** as concurrency rises, throughput should climb steeply (continuous batching), then flatten as the GPU saturates. Past saturation, TTFT p95 grows because requests queue. Your operating point is just before that knee, where throughput is high and p95 latency still meets your SLA.

## 8. Multi-agent RAG patterns on one vLLM server

### 8.1 Shared prefix per agent and a stable prompt layout
Build prompts so each agent's system prompt is byte-identical on every call.

In [ ]:
AGENT_PROMPTS = {
    "router":    "You are a routing agent. Classify the query into: policy, hr, it, other. Reply with one word.",
    "retriever": "You rewrite user questions into concise search queries for a vector database.",
    "answerer":  "You answer strictly from the provided context. Cite doc ids. If unsure, say so.",
}

def build_messages(agent, user_query, context_docs=None):
    msgs = [{"role": "system", "content": AGENT_PROMPTS[agent]}]     # stable first -> prefix cache hit
    if context_docs:
        ctx = "\n\n".join(f"[doc {d['id']}]\n{d['text']}" for d in context_docs)
        msgs.append({"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {user_query}"})
    else:
        msgs.append({"role": "user", "content": user_query})
    return msgs

async def call_agent(agent, query, docs=None, priority=0, max_tokens=256, **kw):
    r = await client.chat.completions.create(
        model="rag-llm", messages=build_messages(agent, query, docs),
        max_tokens=max_tokens, temperature=0.0,
        extra_body={"priority": priority}, **kw)
    return r.choices[0].message.content

print(await call_agent("router", "How many vacation days do I get?", max_tokens=5))

### 8.2 Priorities: protect interactive agents from batch jobs
With `--scheduling-policy priority`, **lower number = higher priority**. Use it so user-facing agents (router, answerer) are not stuck behind background work (re-summarizing, evaluation, indexing).

Priority affects *queue order and preemption* when the server is saturated. It does not speed up a request that is already running.

In [ ]:
async def mixed_workload():
    # 100 low-priority background jobs flood the server...
    bg = [asyncio.create_task(one_request(f"Summarize document {i} in detail.", 256, priority=10))
          for i in range(100)]
    await asyncio.sleep(1.0)
    # ...then 10 interactive requests arrive
    fg = await asyncio.gather(*[one_request(f"Quick question {i}", 64, priority=0) for i in range(10)])
    await asyncio.gather(*bg)
    print("interactive TTFT p50: %.0f ms" % (statistics.median(r["ttft"] for r in fg) * 1000))

await mixed_workload()
# Re-run with priority=10 for the interactive calls too and compare TTFT.

### 8.3 Cap concurrency per agent and add timeouts
Multi-agent graphs can fan out suddenly (one user query becomes 10 sub-agent calls). Use semaphores so one agent can't starve the others, and use timeouts/retries so a stuck request doesn't stall the pipeline.

In [ ]:
AGENT_LIMITS = {"router": asyncio.Semaphore(64), "retriever": asyncio.Semaphore(32), "answerer": asyncio.Semaphore(48)}

async def limited_call(agent, query, docs=None, priority=0, timeout=30, retries=2, **kw):
    for attempt in range(retries + 1):
        try:
            async with AGENT_LIMITS[agent]:
                return await asyncio.wait_for(call_agent(agent, query, docs, priority, **kw), timeout)
        except (asyncio.TimeoutError, Exception) as e:
            if attempt == retries: raise
            await asyncio.sleep(0.5 * 2 ** attempt)

# Parallel fan-out: answer 5 sub-questions at once; the server batches them together
subqs = [f"Sub-question {i} about expense policy" for i in range(5)]
answers = await asyncio.gather(*[limited_call("answerer", q, max_tokens=64) for q in subqs])
print(len(answers), "answers received")

### 8.4 Structured outputs for reliable agent hand-offs
Agents often pass JSON to each other. Constrained decoding guarantees valid JSON and avoids retries, which saves both latency and tokens. vLLM supports JSON-schema constrained output through the OpenAI-compatible API (the exact `extra_body` key can differ by version; recent versions accept `response_format` with a JSON schema).

In [ ]:
route_schema = {
    "type": "object",
    "properties": {"route": {"type": "string", "enum": ["policy", "hr", "it", "other"]},
                   "confidence": {"type": "number"}},
    "required": ["route", "confidence"],
}

r = await client.chat.completions.create(
    model="rag-llm",
    messages=build_messages("router", "My laptop won't boot"),
    max_tokens=60, temperature=0,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "route", "schema": route_schema}},
)
print(r.choices[0].message.content)

### 8.5 One base model, many specialised agents with LoRA
Instead of hosting a separate model per agent, serve one base model and attach small LoRA adapters. vLLM batches requests for different adapters together (multi-LoRA), so the GPU stays shared.

```bash
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --enable-lora --max-loras 4 --max-lora-rank 32 \
  --lora-modules router=/adapters/router answerer=/adapters/answerer
```

Then call with `model="router"` or `model="answerer"`. Much cheaper than separate deployments.

## 9. Cutting latency and freeing memory

### 9.1 Quantization and FP8 KV cache
- **Weights:** AWQ/GPTQ (4-bit) or FP8 shrink weights, leaving more GPU memory for KV cache, so more concurrent requests. FP8 on H100/L40S/Ada GPUs is nearly lossless and also faster.
- **KV cache:** `--kv-cache-dtype fp8` roughly doubles cacheable tokens.

```bash
vllm serve <model> --quantization fp8 --kv-cache-dtype fp8
# or a pre-quantized AWQ checkpoint:
vllm serve TheBloke/...-AWQ --quantization awq
```

Always re-run your RAG eval set after quantizing, since faithfulness to context can shift.

### 9.2 Speculative decoding
A small draft mechanism proposes several tokens; the big model verifies them in one pass. It reduces **inter-token latency at low-to-moderate load** (when the GPU isn't already saturated by batching). At very high concurrency it can reduce throughput, so benchmark. Options include n-gram lookup (no extra model; works well for RAG because answers copy from context) and draft models / EAGLE heads. Configuration is passed through `--speculative-config` in recent versions:

```bash
vllm serve <model> \
  --speculative-config '{"method": "ngram", "num_speculative_tokens": 4, "prompt_lookup_max": 4}'
```

The n-gram method suits RAG especially well because generated answers frequently quote retrieved text.

## 10. Observability: watch these before you tune

vLLM exposes Prometheus metrics at `/metrics`. Key signals:

| Metric (names vary slightly by version) | Meaning | Action |
|---|---|---|
| `vllm:num_requests_waiting` | Queue length | Persistently > 0 means you're saturated; scale out or reduce load |
| `vllm:num_requests_running` | Active batch size | Compare with `max_num_seqs` |
| `vllm:gpu_cache_usage_perc` / `vllm:kv_cache_usage_perc` | KV cache fullness | ~100% with queue = KV-bound: use fp8 KV, shorter context, more GPUs |
| `vllm:num_preemptions_total` | Requests evicted for lack of KV | Rising = reduce `max_num_seqs` / `max_model_len` or add memory |
| `vllm:time_to_first_token_seconds` | TTFT histogram | Prefill / queueing problem |
| `vllm:inter_token_latency_seconds` (older: `time_per_output_token_seconds`) | Decode speed | Batch too large, or chunk budget too high |
| `vllm:prefix_cache_hits_total` / `..._queries_total` | Prefix-cache hit rate | Low hit rate: fix prompt layout |

**Diagnosis cheat sheet**

- Low GPU util, queue empty: not enough concurrent clients; add parallelism in your agents.
- Queue growing, KV usage low: raise `max_num_seqs` / `max_num_batched_tokens`.
- Queue growing, KV usage ~100%: memory-bound; quantize, shrink `max_model_len`, or add GPUs.
- Good throughput but p95 TTFT bad: lower `max_num_batched_tokens`, use priorities, ensure chunked prefill is on.

In [ ]:
import re, requests

def scrape_metrics(url="http://localhost:8000/metrics", keys=("num_requests_running", "num_requests_waiting",
                  "cache_usage_perc", "num_preemptions", "prefix_cache")):
    text = requests.get(url, timeout=5).text
    rows = {}
    for line in text.splitlines():
        if line.startswith("#"): continue
        if line.startswith("vllm:") and any(k in line for k in keys):
            name, _, val = line.rpartition(" ")
            rows[name] = float(val)
    return rows

for k, v in scrape_metrics().items():
    print(f"{v:>14.3f}  {k[:110]}")

In [ ]:
# Live monitor while a load test runs (poll every 2s)
import asyncio, pandas as pd

async def monitor_during(load_coro, interval=2.0):
    samples, done = [], asyncio.Event()
    async def poll():
        while not done.is_set():
            m = scrape_metrics()
            samples.append({
                "running": next((v for k, v in m.items() if "num_requests_running" in k), None),
                "waiting": next((v for k, v in m.items() if "num_requests_waiting" in k), None),
                "kv_usage": next((v for k, v in m.items() if "cache_usage_perc" in k), None)})
            await asyncio.sleep(interval)
    t = asyncio.create_task(poll())
    await load_coro
    done.set(); await t
    return pd.DataFrame(samples)

df = await monitor_during(load_test(64, n_requests=200))
df.plot(subplots=True, figsize=(7, 5), title="vLLM server state during load"); plt.show()

## 11. Benchmarking and the tuning checklist

Use vLLM's own benchmark tool against your running server (it replays realistic request distributions):

```bash
# newer versions
vllm bench serve --backend openai-chat --base-url http://localhost:8000 \
  --endpoint /v1/chat/completions --model rag-llm \
  --dataset-name random --random-input-len 3000 --random-output-len 200 \
  --num-prompts 500 --request-rate 8
# older versions: python benchmarks/benchmark_serving.py ...
```

Match input/output lengths to your real RAG traffic (retrieved context makes inputs long: often 2-6k tokens in, 100-400 out).

### Tuning loop
1. **Baseline** with realistic traffic and record tok/s, TTFT p50/p95, ITL p50/p95, KV usage, preemptions.
2. Change **one** knob; re-measure.
3. Keep changes that improve your SLA metric without breaking another.

### Production checklist
- [ ] One shared vLLM server (or one replica per GPU behind a load balancer), not a model per agent
- [ ] Stable prompt prefixes first, variable content last; prefix caching on
- [ ] Chunked prefill on; `max_num_batched_tokens` tuned (2k-4k interactive, 8k+ batch)
- [ ] `max_model_len` as small as your use case allows; cap retrieved context (top-k, token budget)
- [ ] `gpu_memory_utilization` 0.88-0.92 with GPU dedicated to vLLM
- [ ] FP8/AWQ weights and/or FP8 KV cache if memory-bound (re-validate quality)
- [ ] Async clients with bounded concurrency, timeouts, retries, backpressure (429/queue limits)
- [ ] Priority scheduling for interactive vs batch agents
- [ ] Structured outputs for agent-to-agent JSON
- [ ] Prometheus + Grafana dashboards and alerts on queue length, KV usage, preemptions, TTFT p95
- [ ] Warm-up requests on startup (CUDA graph capture/compile and prefix cache) before accepting traffic
- [ ] Pin vLLM version and the model revision; test upgrades on a staging replica
- [ ] Health check on `/health`; auto-restart (systemd/Docker/Kubernetes)

### Clean shutdown

In [ ]:
server.terminate()
try:
    server.wait(timeout=30)
except Exception:
    server.kill()
print("server stopped")